# sh: Manejo de Excepciones

## Metaprogramación - type

![objeto-clase-metaclase](exc_media/instance-of.png)

## Clases Protagonistas


In [1]:
class ErrorReturnCodeMeta(type):
	"""
	a metaclass which provides the ability for an ErrorReturnCode (or
	derived) instance, imported from one sh module, to be considered the
	subclass of ErrorReturnCode from another module.
	"""
	def __subclasscheck__(self, o):
		other_bases = {b.__name__ for b in o.__bases__}
		return self.__name__ in other_bases or o.__name__ == self.__name__

In [2]:
class ErrorReturnCode(Exception):
	__metaclass__ = ErrorReturnCodeMeta
	# Resto de la clase

## Dado un código de error, ¿qué excepción *throweamos*?

*ErrorReturnCode* y *ErrorReturnCodeMeta* no se instancian **nunca**.

![jerarquía](exc_media/Sh_module.png)

## ¿Qué recorrido hace la información?

In [ ]:
# Dentro del código correspondiente al proceso padre
                def fn(exit_code):
                    with process_assign_lock:
                        return self.command.handle_command_exit_code(exit_code)

                handle_exit_code = fn

### ... donde este método se define de la siguiente manera:

In [ ]:
def handle_command_exit_code(self, code):
"""
Vemos si se produjo una excepción o recibimos un código de error inesperado. 
De ser así, creamos y levantamos la excepción correspondiente
"""
	ca = self.call_args # call_args se define en Command
	exc_class = get_exc_exit_code_would_raise(code, ca["ok_code"], ca["piped"])
	if exc_class:
		exc = exc_class(
			self.ran, 
			self.process.stdout, 
			self.process.stderr,
			ca["truncate_exc"]
			)
		raise exc

### Tirando del hilo...

In [ ]:
def get_exc_exit_code_would_raise(exit_code, ok_codes, sigpipe_ok):
	exc = None
	success = exit_code in ok_codes
	bad_sig = -exit_code in SIGNALS_THAT_SHOULD_THROW_EXCEPTION
	
	# if this is a piped command, SIGPIPE must be ignored by us and not raise an
	# exception, since it's perfectly normal for the consumer of a process's
	# pipe to terminate early
	
	if sigpipe_ok and -exit_code == signal.SIGPIPE:
		bad_sig = False
		success = True
	
	if not success or bad_sig:
		exc = get_rc_exc(exit_code)
	return exc

In [ ]:
def get_rc_exc(rc):
"""
takes a exit code or negative signal number and produces an exception
that corresponds to that return code. positive return codes yield
ErrorReturnCode exception, negative return codes yield SignalException
we also cache the generated exception so that only one signal of that type
exists, preserving identity
"""
	try:
		return rc_exc_cache[rc]
	except KeyError:
		pass
	
	if rc >= 0:
		name = f"ErrorReturnCode_{rc}"
		base = ErrorReturnCode
	
	else:
		name = f"SignalException_{SIGNAL_MAPPING[abs(rc)]}"
		base = SignalException
	
	exc = ErrorReturnCodeMeta(name, (base,), {"exit_code": rc})
	rc_exc_cache[rc] = exc
	return exc

### ... donde

In [ ]:
rc_exc_cache: Dict[str, Type[ErrorReturnCode]] = {}